<a href="https://colab.research.google.com/github/Mobinmo83/AMT/blob/main/piano_amt/notebooks/Demo/demo_assets_and_checkpoint_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Public Demo Setup / Initialization Notebook

This notebook is **not** the public demo itself. It is the **private preparation notebook** you run once to:

1. point the repo at your Hugging Face model repo
2. export a tiny public MAESTRO demo subset from your private Drive/cache
3. upload the public checkpoint + demo assets to Hugging Face
4. sanity-check that the public demo layer can download and load the checkpoint

Use this notebook whenever you want to refresh the public checkpoint or update the curated sample set.


In [11]:

#@title 0. Setup repo access and install helper dependencies
from pathlib import Path


REPO_URL = "https://github.com/Mobinmo83/AMT.git"
REPO_BRANCH = "main"
REPO_CLONE_DIR = "/content/repo"

repo_dir = Path(REPO_CLONE_DIR)
if not repo_dir.exists():
    !git clone -q --branch "$REPO_BRANCH" "$REPO_URL" "$REPO_CLONE_DIR"
else:
    print(f"Repo already present at {repo_dir}")

%cd $REPO_CLONE_DIR
!pip -q install -r piano_amt/requirements_demo.txt huggingface_hub





Repo already present at /content/repo
/content/repo
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 33.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.7/90.7 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.7/253.7 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 65.3 MB/s eta 0:00:00


In [14]:
#@title 1. Imports
import json
import os
import sys
from pathlib import Path

from huggingface_hub import notebook_login

PROJECT_ROOT = str(Path(REPO_CLONE_DIR) / "piano_amt")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from demo.export_demo_assets import export_demo_assets
from demo.upload_to_hf import upload_demo_package


## 2. Hugging Face authentication

Run the next cell and log in with a Hugging Face token that has permission to create or update your model repo.


In [15]:

#@title 2. Log in to Hugging Face
notebook_login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [22]:
import pandas as pd
from pathlib import Path

csv_path = Path(MAESTRO_ROOT) / "maestro-v3.0.0.csv"
df = pd.read_csv(csv_path)

test_df = df[df["split"] == "test"].copy()

print("Number of test files:", len(test_df))
print("\nFirst 30 test stems:\n")

for i, row in test_df.head(30).iterrows():
    print(Path(row["audio_filename"]).stem)

Number of test files: 177

First 30 test stems:

MIDI-Unprocessed_11_R1_2009_06-09_ORIG_MID--AUDIO_11_R1_2009_11_R1_2009_07_WAV
MIDI-Unprocessed_02_R1_2009_03-06_ORIG_MID--AUDIO_02_R1_2009_02_R1_2009_04_WAV
MIDI-Unprocessed_01_R1_2006_01-09_ORIG_MID--AUDIO_01_R1_2006_04_Track04_wav
MIDI-Unprocessed_24_R1_2006_01-05_ORIG_MID--AUDIO_24_R1_2006_03_Track03_wav
MIDI-Unprocessed_11_R1_2009_06-09_ORIG_MID--AUDIO_11_R1_2009_11_R1_2009_08_WAV
MIDI-Unprocessed_Recital13-15_MID--AUDIO_15_R1_2018_wav--1
MIDI-UNPROCESSED_09-10_R1_2014_MID--AUDIO_09_R1_2014_wav--4
MIDI-Unprocessed_11_R1_2009_01-05_ORIG_MID--AUDIO_11_R1_2009_11_R1_2009_04_WAV
MIDI-Unprocessed_SMF_17_R1_2004_03-06_ORIG_MID--AUDIO_20_R2_2004_12_Track12_wav--1
MIDI-Unprocessed_04_R3_2011_MID--AUDIO_R3-D2_05_Track05_wav
MIDI-Unprocessed_05_R1_2008_01-04_ORIG_MID--AUDIO_05_R1_2008_wav--3
MIDI-Unprocessed_R2_D2-12-13-15_mid--AUDIO-from_mp3_12_R2_2015_wav--3
MIDI-Unprocessed_Recital16_MID--AUDIO_16_R1_2018_wav--3
MIDI-Unprocessed_03_R1_2006

In [23]:
#@title 3. Private paths and public repo settings (edit these)
from pathlib import Path

HF_REPO_ID = "Mobinmo83/piano-amt-demo"
PUBLIC_PRIVATE = False

CHECKPOINT_PATH = "/content/drive/MyDrive/piano_amt/runs/OF_full_run_v2/checkpoints/best.pt"
MAESTRO_ROOT = "/content/drive/MyDrive/piano_amt/maestro-v3.0.0"
CACHE_DIR = "/content/drive/MyDrive/piano_amt/cache"
OUTPUT_DEMO_ASSETS = str(Path(REPO_CLONE_DIR) / "piano_amt" / "demo_assets")

# Add 2–5 short stems from your private MAESTRO test set.
DEMO_STEMS = [
    "MIDI-Unprocessed_Recital13-15_MID--AUDIO_15_R1_2018_wav--1",
    "MIDI-Unprocessed_XP_04_R1_2004_03-05_ORIG_MID--AUDIO_04_R1_2004_06_Track06_wav",
    "MIDI-Unprocessed_14_R1_2008_01-05_ORIG_MID--AUDIO_14_R1_2008_wav--3",
]

checkpoint_path = Path(CHECKPOINT_PATH)
maestro_root = Path(MAESTRO_ROOT)
cache_dir = Path(CACHE_DIR)
output_demo_assets = Path(OUTPUT_DEMO_ASSETS)
manifest_path = Path(REPO_CLONE_DIR) / "piano_amt" / "demo" / "sample_manifest.json"

print("HF repo:", HF_REPO_ID)
print("Checkpoint:", checkpoint_path)
print("MAESTRO root:", maestro_root)
print("Cache dir:", cache_dir)
print("Demo assets dir:", output_demo_assets)
print("Manifest path:", manifest_path)
print("Demo stems:", DEMO_STEMS)

HF repo: Mobinmo83/piano-amt-demo
Checkpoint: /content/drive/MyDrive/piano_amt/runs/OF_full_run_v2/checkpoints/best.pt
MAESTRO root: /content/drive/MyDrive/piano_amt/maestro-v3.0.0
Cache dir: /content/drive/MyDrive/piano_amt/cache
Demo assets dir: /content/repo/piano_amt/demo_assets
Manifest path: /content/repo/piano_amt/demo/sample_manifest.json
Demo stems: ['MIDI-Unprocessed_Recital13-15_MID--AUDIO_15_R1_2018_wav--1', 'MIDI-Unprocessed_XP_04_R1_2004_03-05_ORIG_MID--AUDIO_04_R1_2004_06_Track06_wav', 'MIDI-Unprocessed_14_R1_2008_01-05_ORIG_MID--AUDIO_14_R1_2008_wav--3']



## 4. Export a tiny public MAESTRO demo subset

This reads your **private** MAESTRO CSV/cache and writes only a few public demo files into your repo:

- `demo_assets/sample_01.wav`
- `demo_assets/sample_01_labels.npz`
- ...
- `demo/sample_manifest.json`


In [20]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [24]:

#@title 4. Export demo assets from your private dataset/cache
export_demo_assets(
    maestro_root=maestro_root,
    cache_dir=cache_dir,
    output_dir=output_demo_assets,
    names=DEMO_STEMS,
)

print("\nManifest contents:\n")
print((manifest_path).read_text())


Wrote manifest → /content/repo/piano_amt/demo/sample_manifest.json

Manifest contents:

{
  "samples": [
    {
      "name": "MAESTRO Test 01 \u2014 MIDI-Unprocessed_Recital13-15_MID--AUDIO_15_R1_2018_wav--1",
      "audio": "demo_assets/sample_01.wav",
      "labels": "demo_assets/sample_01_labels.npz"
    },
    {
      "name": "MAESTRO Test 02 \u2014 MIDI-Unprocessed_XP_04_R1_2004_03-05_ORIG_MID--AUDIO_04_R1_2004_06_Track06_wav",
      "audio": "demo_assets/sample_02.wav",
      "labels": "demo_assets/sample_02_labels.npz"
    },
    {
      "name": "MAESTRO Test 03 \u2014 MIDI-Unprocessed_14_R1_2008_01-05_ORIG_MID--AUDIO_14_R1_2008_wav--3",
      "audio": "demo_assets/sample_03.wav",
      "labels": "demo_assets/sample_03_labels.npz"
    }
  ]
}



## 5. Upload checkpoint + demo assets to Hugging Face

This creates or updates the public model repo and uploads:

- `checkpoints/best.pt`
- `demo_assets/*`
- `demo/sample_manifest.json`


In [25]:

#@title 5. Upload the public demo package to Hugging Face
upload_demo_package(
    repo_id=HF_REPO_ID,
    checkpoint=checkpoint_path,
    demo_assets=output_demo_assets,
    manifest=manifest_path,
    private=PUBLIC_PRIVATE,
)


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...un_v2/checkpoints/best.pt:   0%|          |  551kB /  318MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

  ...demo_assets/sample_03.wav:  89%|########8 | 27.2MB / 30.5MB            

  ...sets/sample_01_labels.npz:  36%|###6      | 34.1kB / 93.4kB            

  ...sets/sample_02_labels.npz:  36%|###6      | 14.4kB / 39.6kB            

  ...sets/sample_03_labels.npz:  36%|###6      | 16.8kB / 46.1kB            

Uploaded checkpoint + demo assets to Hugging Face repo: Mobinmo83/piano-amt-demo



## 6. Point the repo demo layer at the public HF repo

You can either:

1. edit `demo/demo_config.py` permanently, or
2. set environment variables inside the public notebook.

The public notebook generated for this repo uses environment variables in a config cell, so you usually do **not** need to hard-edit the Python file each time.


In [26]:

#@title 6. Optional sanity check: test public checkpoint download/load
os.environ["AMT_DEMO_HF_REPO_ID"] = HF_REPO_ID
os.environ["AMT_DEMO_HF_CHECKPOINT_FILENAME"] = "checkpoints/best.pt"
os.environ["AMT_DEMO_REPO_ROOT"] = REPO_CLONE_DIR

from demo.checkpoint import load_demo_model, format_checkpoint_summary

model, ckpt, ckpt_path = load_demo_model("cpu")
print(format_checkpoint_summary(model, ckpt, ckpt_path))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


checkpoints/best.pt:   0%|          | 0.00/318M [00:00<?, ?B/s]

Checkpoint: /content/amt_demo/checkpoints/checkpoints/best.pt
Epoch: 990
Val loss: 0.045293346999117925
Best val loss: 0.045293346999117925
Global step: 118800
Trainable params: 26,491,256
